# 11_check_thresholds

正例データと負例データを使って RC / CC / MC の top1 スコア分布を確認し、誤判定数が最小になる閾値候補を探索する。

## 方針

論文と同様に、正例データと負例データのスコア分布から、両者を最もよく分離する閾値を探す。

この notebook では以下を行う。

1. 正例データと負例データを全件評価する
2. 各方式の top1 スコアを収集する
3. 候補閾値を走査し、誤判定数が最小の点を探す
4. 結果をテキストとグラフで確認する

In [ ]:
# Notebook から src 配下の自作モジュールを import できるようにする設定
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
# グラフ表示 backend を設定する
try:
    get_ipython().run_line_magic("matplotlib", "qt")
    print("matplotlib backend: qt (別ウィンドウ表示)")
except Exception as exc:
    get_ipython().run_line_magic("matplotlib", "inline")
    print("matplotlib backend: inline (notebook 内表示)")
    print(f"qt backend は使えませんでした: {exc}")


In [ ]:
# 設定値・評価関数・描画ライブラリを読み込む
import matplotlib.pyplot as plt

from tanigawa_shoshi.config import NEGATIVE_EXAMPLES_PATH, POSITIVE_EXAMPLES_PATH
from tanigawa_shoshi.evaluation import DEFAULT_THRESHOLDS, evaluate_dataset
from tanigawa_shoshi.evaluation_data import load_examples

plt.rcParams["font.family"] = ["Hiragino Sans", "Yu Gothic", "IPAexGothic", "Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 閾値確認の対象データと検索条件を設定する
positive_examples_path = POSITIVE_EXAMPLES_PATH
negative_examples_path = NEGATIVE_EXAMPLES_PATH
rows = 100
paper_thresholds = DEFAULT_THRESHOLDS.copy()

print(f"positive examples path: {positive_examples_path}")
print(f"negative examples path: {negative_examples_path}")
print(f"rows: {rows}")
print("paper thresholds")
for score_name in ["rc", "cc", "mc"]:
    print(f"  {score_name}: {paper_thresholds[score_name]}")


In [ ]:
# 正例データと負例データを読み込む
positive_examples = load_examples(positive_examples_path)
negative_examples = load_examples(negative_examples_path)

print(f"positive count: {len(positive_examples)}")
print(f"negative count: {len(negative_examples)}")


In [ ]:
# 正例データと負例データを全件評価する
# 約30秒
positive_evaluation = evaluate_dataset(
    positive_examples,
    paper_thresholds,
    rows=rows,
    mode="positive",
)
negative_evaluation = evaluate_dataset(
    negative_examples,
    paper_thresholds,
    rows=rows,
    mode="negative",
)

print("threshold search input evaluation finished")


In [ ]:
# 閾値探索と表形式出力の補助関数
SCORE_NAMES = ["rc", "cc", "mc"]
SCORE_LABELS = {"rc": "RC", "cc": "CC", "mc": "MC"}
METHOD_ORDER = ["bm25", "rc", "cc", "mc"]
METHOD_LABELS = {
    "bm25": "BM25",
    "rc": "RC",
    "cc": "CC",
    "mc": "MC",
}


def extract_positive_score_samples(evaluation_result, score_name):
    samples = []
    for result in evaluation_result["results"]:
        top_candidate = result["method_results"][score_name]["top_candidate"]
        score = None if top_candidate is None else top_candidate[score_name]
        is_correct = bool(top_candidate and result["doi"] and top_candidate["doi"] == result["doi"])
        samples.append(
            {
                "example_id": result["example_id"],
                "score": score,
                "is_correct": is_correct,
            }
        )
    return samples


def extract_negative_score_samples(evaluation_result, score_name):
    samples = []
    for result in evaluation_result["results"]:
        top_candidate = result["method_results"][score_name]["top_candidate"]
        score = None if top_candidate is None else top_candidate[score_name]
        samples.append(
            {
                "example_id": result["example_id"],
                "score": score,
            }
        )
    return samples


def build_threshold_candidates(positive_samples, negative_samples):
    score_values = []
    for sample in positive_samples + negative_samples:
        score = sample["score"]
        if score is not None:
            score_values.append(float(score))
    if not score_values:
        return [0.0]
    unique_scores = sorted(set(score_values))
    threshold_candidates = [0.0]
    threshold_candidates.extend(unique_scores)
    if unique_scores[-1] < 1.0:
        threshold_candidates.append(1.0)
    return threshold_candidates


def evaluate_threshold(positive_samples, negative_samples, threshold):
    detected_count = 0
    missed_count = 0
    positive_false_positive_count = 0
    true_negative_count = 0
    negative_false_positive_count = 0

    for sample in positive_samples:
        score = sample["score"]
        is_correct = sample["is_correct"]
        if score is None:
            missed_count += 1
        elif score >= threshold:
            if is_correct:
                detected_count += 1
            else:
                positive_false_positive_count += 1
        else:
            missed_count += 1

    for sample in negative_samples:
        score = sample["score"]
        if score is None or score < threshold:
            true_negative_count += 1
        else:
            negative_false_positive_count += 1

    total_error_count = missed_count + positive_false_positive_count + negative_false_positive_count
    total_count = len(positive_samples) + len(negative_samples)
    total_error_rate = (total_error_count / total_count * 100) if total_count else 0.0

    return {
        "threshold": threshold,
        "detected_count": detected_count,
        "missed_count": missed_count,
        "positive_false_positive_count": positive_false_positive_count,
        "true_negative_count": true_negative_count,
        "negative_false_positive_count": negative_false_positive_count,
        "total_error_count": total_error_count,
        "total_error_rate": total_error_rate,
    }


def find_best_threshold(positive_samples, negative_samples):
    threshold_candidates = build_threshold_candidates(positive_samples, negative_samples)
    evaluations = [
        evaluate_threshold(positive_samples, negative_samples, threshold)
        for threshold in threshold_candidates
    ]
    best_result = min(
        evaluations,
        key=lambda item: (
            item["total_error_count"],
            item["negative_false_positive_count"],
            -item["detected_count"],
            item["threshold"],
        ),
    )
    return best_result, evaluations


def format_count_ratio(value, total_count):
    ratio = (value / total_count * 100) if total_count else 0.0
    return f"{value:,} ({ratio:.2f}%)"


def print_text_table(title, headers, rows):
    widths = [len(header) for header in headers]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(str(value)))

    def render_row(row_values):
        return " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(row_values))

    separator = "-+-".join("-" * width for width in widths)

    print("=" * 100)
    print(title)
    print(render_row(headers))
    print(separator)
    for row in rows:
        print(render_row(row))


def build_positive_table_rows(summary):
    total_count = summary["input_count"]
    rows = []
    for method_name in METHOD_ORDER:
        method_summary = summary["methods"].get(method_name, {})
        rows.append(
            [
                METHOD_LABELS[method_name],
                format_count_ratio(method_summary.get("detected_count", 0), total_count),
                format_count_ratio(method_summary.get("missed_count", 0), total_count),
                format_count_ratio(method_summary.get("false_positive_count", 0), total_count),
            ]
        )
    return rows


def build_negative_table_rows(summary):
    total_count = summary["input_count"]
    rows = []
    for method_name in METHOD_ORDER:
        method_summary = summary["methods"].get(method_name, {})
        rows.append(
            [
                METHOD_LABELS[method_name],
                format_count_ratio(method_summary.get("true_negative_count", 0), total_count),
                format_count_ratio(method_summary.get("false_positive_count", 0), total_count),
            ]
        )
    return rows

In [ ]:
# RC / CC / MC ごとに最適閾値候補を探索し、その閾値で再評価する
threshold_search_results = {}

for score_name in SCORE_NAMES:
    positive_samples = extract_positive_score_samples(positive_evaluation, score_name)
    negative_samples = extract_negative_score_samples(negative_evaluation, score_name)
    best_result, evaluation_trace = find_best_threshold(positive_samples, negative_samples)
    threshold_search_results[score_name] = {
        "positive_samples": positive_samples,
        "negative_samples": negative_samples,
        "best_result": best_result,
        "evaluation_trace": evaluation_trace,
        "paper_threshold": paper_thresholds[score_name],
    }

optimized_thresholds = {
    score_name: threshold_search_results[score_name]["best_result"]["threshold"]
    for score_name in SCORE_NAMES
}

optimized_positive_evaluation = evaluate_dataset(
    positive_examples,
    optimized_thresholds,
    rows=rows,
    mode="positive",
)
optimized_negative_evaluation = evaluate_dataset(
    negative_examples,
    optimized_thresholds,
    rows=rows,
    mode="negative",
)

print("threshold search finished")
print("optimized thresholds")
for score_name in SCORE_NAMES:
    print(f"  {score_name}: {optimized_thresholds[score_name]:.4f}")

In [ ]:
# 論文閾値と最適化閾値の結果を表形式で表示する
print_text_table(
    "論文閾値での正例評価",
    ["スコアリング", "正常検出", "検出漏れ", "誤検出"],
    build_positive_table_rows(positive_evaluation["summary"]),
)
print()
print_text_table(
    "論文閾値での負例評価",
    ["スコアリング", "正しく非同定", "誤検出"],
    build_negative_table_rows(negative_evaluation["summary"]),
)
print()
print_text_table(
    "最適化閾値での正例評価",
    ["スコアリング", "正常検出", "検出漏れ", "誤検出"],
    build_positive_table_rows(optimized_positive_evaluation["summary"]),
)
print()
print_text_table(
    "最適化閾値での負例評価",
    ["スコアリング", "正しく非同定", "誤検出"],
    build_negative_table_rows(optimized_negative_evaluation["summary"]),
)

In [ ]:
# スコア分布をヒストグラムで表示する
fig, axes = plt.subplots(3, 1, figsize=(12, 16), constrained_layout=True)

for axis, score_name in zip(axes, SCORE_NAMES):
    positive_scores = [sample['score'] for sample in threshold_search_results[score_name]['positive_samples'] if sample['score'] is not None]
    negative_scores = [sample['score'] for sample in threshold_search_results[score_name]['negative_samples'] if sample['score'] is not None]
    best_threshold = threshold_search_results[score_name]['best_result']['threshold']
    paper_threshold = threshold_search_results[score_name]['paper_threshold']

    axis.hist(positive_scores, bins=20, alpha=0.6, label='正例', color='#4daf4a')
    axis.hist(negative_scores, bins=20, alpha=0.6, label='負例', color='#e41a1c')
    axis.axvline(best_threshold, color='#377eb8', linestyle='--', linewidth=2, label=f'最適閾値 {best_threshold:.4f}')
    axis.axvline(paper_threshold, color='#984ea3', linestyle=':', linewidth=2, label=f'論文閾値 {paper_threshold:.4f}')
    axis.set_title(f"{SCORE_LABELS[score_name]} の top1 スコア分布")
    axis.set_xlabel('スコア')
    axis.set_ylabel('件数')
    axis.grid(axis='y', alpha=0.3)
    axis.legend(loc='upper center', ncol=2)

fig.suptitle('正例・負例の top1 スコア分布', fontsize=16)
plt.show()


In [ ]:
# 閾値ごとの誤判定数を折れ線で表示する
fig, axes = plt.subplots(3, 1, figsize=(12, 16), constrained_layout=True)

for axis, score_name in zip(axes, SCORE_NAMES):
    evaluation_trace = threshold_search_results[score_name]['evaluation_trace']
    thresholds = [item['threshold'] for item in evaluation_trace]
    total_errors = [item['total_error_count'] for item in evaluation_trace]
    missed_counts = [item['missed_count'] for item in evaluation_trace]
    negative_false_positives = [item['negative_false_positive_count'] for item in evaluation_trace]
    best_threshold = threshold_search_results[score_name]['best_result']['threshold']

    axis.plot(thresholds, total_errors, label='総誤判定数', color='#377eb8')
    axis.plot(thresholds, missed_counts, label='検出漏れ', color='#ff7f00', alpha=0.8)
    axis.plot(thresholds, negative_false_positives, label='負例側誤検出', color='#e41a1c', alpha=0.8)
    axis.axvline(best_threshold, color='#4daf4a', linestyle='--', linewidth=2, label=f'最適閾値 {best_threshold:.4f}')
    axis.set_title(f"{SCORE_LABELS[score_name]} の閾値と誤判定数")
    axis.set_xlabel('閾値')
    axis.set_ylabel('件数')
    axis.grid(alpha=0.3)
    axis.legend(loc='upper right')

fig.suptitle('閾値ごとの誤判定数の変化', fontsize=16)
plt.show()
